# Hito 1 — Baseline de recuperación dispersa (TF-IDF y BM25)

**Proyecto Final · Procesamiento del Lenguaje Natural · Máster en IA Aplicada (UC3M)**

Este notebook implementa el baseline del Hito 1:

1. Carga del **corpus** y preprocesado con **spaCy** (lematización, filtrado POS, stopwords, frecuencia documental).
2. Conjunto de **consultas de evaluación** *known-item* (`queries.jsonl`).
3. Baselines de recuperación **dispersa**: **TF-IDF + coseno** y **BM25**.
4. Evaluación con **Recall@K (K=5, 10)** y **MRR**, y análisis de los fallos.

> **Corpus**: este notebook se ejecuta sobre el corpus **Food.com** (`data/corpus_foodcom.jsonl`,
> 10.000 recetas, muestreadas de forma estratificada por categoría para asegurar variedad).
> Las consultas (`data/queries_foodcom.jsonl`) se generaron con `llama3.1:8b`. Para usar otro
> corpus, cambiad la celda de configuración. *(Se incluye también un mini-corpus de demo,
> `data/corpus_demo.jsonl`, por si se quiere una ejecución rápida sin el dataset.)*


## 0. Configuración

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

# === CONFIGURACIÓN — cambia esto para usar otro corpus ===
CORPUS_PATH   = "data/corpus_foodcom.jsonl"  # JSONL con {"doc_id","text"} por línea
QUERIES_PATH  = "data/queries_foodcom.jsonl" # JSONL con {"qid","query","relevant_ids"}
SPACY_MODEL   = "en_core_web_sm"             # es_core_news_sm para corpus en español
KS            = (5, 10)                        # valores de K para Recall@K
BM25_K1, BM25_B = 1.5, 0.75                   # parámetros BM25 (repórtalos en el MILESTONE)
MIN_DF        = 2                             # frecuencia documental mínima de un término
MAX_DF_RATIO  = 0.5                           # ratio máximo de documentos (evita términos ubicuos)
# =======================================================
print("Config OK")

Config OK


## 1. Carga del corpus

In [2]:
from src.corpus import load_jsonl
docs = load_jsonl(CORPUS_PATH)
doc_ids = [d["doc_id"] for d in docs]
print(f"{len(docs)} documentos cargados")
print("Ejemplo:", docs[0]["doc_id"], "->", docs[0]["text"][:120], "...")

10000 documentos cargados
Ejemplo: recipe_60627 -> Rabbit Paprika
Ingredients: butter, red onion, bay leaf, sweet paprika, caraway seed, white pepper, black pepper, cayenn ...


## 2. Preprocesado con spaCy

La **misma** función se aplica a documentos y a consultas (requisito del enunciado:
*"las consultas deben preprocesarse exactamente igual que los documentos"*).
Pasos: tokenización, **lematización**, filtrado por **categoría gramatical** (nos
quedamos con sustantivos, nombres propios, verbos, adjetivos y adverbios),
eliminación de **stopwords** y de tokens no alfabéticos.

In [3]:
from src.preprocess import load_nlp, preprocess, preprocess_many
nlp = load_nlp(SPACY_MODEL)
corpus_tokens = preprocess_many([d["text"] for d in docs], nlp)

ejemplo = docs[0]
print("ORIGINAL:", ejemplo["text"])
print("LEMAS   :", preprocess(ejemplo["text"], nlp))

ORIGINAL: Rabbit Paprika
Ingredients: butter, red onion, bay leaf, sweet paprika, caraway seed, white pepper, black pepper, cayenne pepper, rabbit, salt, sour cream
Instructions: Melt 4 tblsps of the butter in a Dutch oven over high heat. When foam subsides, add onion slices, Saute, stirring frequently, until onion begins to wilt, about 2 minutes. Add 1/3 cup of stock and bay leaf; heat to boiling. Reduce heat to very low; cook, uncovered, stirring occasionally, for 30 minutes. Combine 1 tsp of paprika, caraway seeds and 3 peppers in a small bowl. Melt remaining 3 tblsps butter in large skillet over medium-high heat. When foam subsides, saute rabbit, in 2 batches if necessary, 3 minutes on each side. Sprinkle pieces evenly with spice mixture. Reduce heat to medium; continue to saute until browned, about 5 minutes. Transfer rabbit to casserole with onions. Pour remaining stock in skillet; heat to boiling, scraping bottom and sides of skillet to loosen browned bits. Add contents of skill

## 3. Vocabulario por frecuencia documental

Descartamos términos demasiado **raros** (df < `MIN_DF`) y demasiado **comunes**
(df > `MAX_DF_RATIO`·N). Documentos y consultas se restringen a este vocabulario,
de modo que las palabras fuera de vocabulario (OOV) se ignoran en ambos baselines.

In [4]:
from src.baselines import build_vocabulary, apply_vocabulary, document_frequencies
vocab = build_vocabulary(corpus_tokens, min_df=MIN_DF, max_df_ratio=MAX_DF_RATIO)
corpus_tokens_f = [apply_vocabulary(t, vocab) for t in corpus_tokens]
print(f"Vocabulario: {len(vocab)} términos")

# La misma tubería para una consulta:
def pp(text):
    return apply_vocabulary(preprocess(text, nlp), vocab)

import collections
df = document_frequencies(corpus_tokens)
print("Términos más frecuentes:", df.most_common(8))

Vocabulario: 5842 términos
Términos más frecuentes: [('instruction', 9988), ('minute', 7136), ('add', 6714), ('ingredient', 5945), ('salt', 5847), ('ingredients', 5654), ('pepper', 5167), ('heat', 5075)]


## 4. Baselines de recuperación dispersa

In [5]:
from src.baselines import TfidfRetriever, BM25Retriever
tfidf = TfidfRetriever(doc_ids).fit(corpus_tokens_f)
bm25  = BM25Retriever(doc_ids, k1=BM25_K1, b=BM25_B).fit(corpus_tokens_f)
print("TF-IDF: vocabulario del vectorizador =", len(tfidf.vectorizer.vocabulary_))
print(f"BM25  : k1={BM25_K1}, b={BM25_B} (implementación: rank_bm25.BM25Okapi)")

# Ejemplo de búsqueda:
q = "chicken curry with coconut milk and rice"
print("\nConsulta:", q)
print("TF-IDF  ->", tfidf.search(pp(q), 3))
print("BM25    ->", bm25.search(pp(q), 3))

TF-IDF: vocabulario del vectorizador = 5842
BM25  : k1=1.5, b=0.75 (implementación: rank_bm25.BM25Okapi)

Consulta: chicken curry with coconut milk and rice
TF-IDF  -> [('recipe_458700', 0.6431417576608345), ('recipe_74912', 0.6061453649891193), ('recipe_407690', 0.5114079965741846)]
BM25    -> [('recipe_541331', 20.620360014779898), ('recipe_37041', 19.123431367928912), ('recipe_74912', 18.801736245947637)]


## 5. Conjunto de consultas de evaluación (`queries.jsonl`)

Consultas *known-item*: para documentos elegidos al azar, un **LLM** redacta la
consulta que haría un usuario para encontrarlos, **sin copiar frases**. El
documento de origen es el relevante. Formato: `{"qid","query","relevant_ids"}`.

> Aquí cargamos un `queries.jsonl` ya preparado. Para **generarlo con un LLM**
> sobre tu corpus real, usa `src/generate_queries.py` (celda siguiente, desactivada
> por defecto). **Recuerda revisar ≥10 consultas a mano** y documentarlo.

In [6]:
from src.generate_queries import load_queries
queries = load_queries(QUERIES_PATH)
print(f"{len(queries)} consultas cargadas")
for q in queries[:3]:
    print(" ", q["qid"], "|", q["query"], "->", q["relevant_ids"])

34 consultas cargadas
  q01 | Coconut based no bake energy bars with flax seed and dark chocolate topping -> ['recipe_479373']
  q02 | Fried Jewish fish patties made with ground fish and matzo meal, served on a bed of greens. -> ['recipe_60179']
  q03 | Grilled Middle Eastern style chicken kebabs with vegetables marinating in a spicy citrus sauce -> ['recipe_170046']


### 5.b (Opcional) Generar las consultas con un LLM

Pon `RUN_LLM = True` y configura el endpoint. Con Ollama local usa
`model="qwen2.5:1.5b"`; con el servidor de la UC3M usa
`url="https://yiyuan.tsc.uc3m.es/api/generate"`, `model="llama3.1:8b"` y tu
`api_key`. Genera unas 35 para poder descartar en la revisión manual y quedarte
con ≥30.

In [7]:
RUN_LLM = False  # ponlo a True para generar consultas con el LLM

if RUN_LLM:
    from src.generate_queries import generate_queries, print_for_review, save_queries
    gen = generate_queries(
        docs, n_queries=35, seed=11235,
        model="qwen2.5:1.5b",
        url="http://localhost:11434/api/generate",
        # api_key="sk-alumnos-...",  # para el endpoint de la UC3M
    )
    print_for_review(gen, n=10)          # REVISIÓN MANUAL (≥10): documenta los descartes
    save_queries(gen, "data/queries_generadas.jsonl")
else:
    print("Generación por LLM desactivada (RUN_LLM=False). Usando queries.jsonl precargado.")

Generación por LLM desactivada (RUN_LLM=False). Usando queries.jsonl precargado.


## 6. Evaluación: Recall@K y MRR

In [8]:
import pandas as pd
from src.evaluation import evaluate

s_tfidf, r_tfidf = evaluate(tfidf, queries, pp, ks=KS)
s_bm25,  r_bm25  = evaluate(bm25,  queries, pp, ks=KS)

tabla = pd.DataFrame([s_tfidf, s_bm25], index=["TF-IDF + coseno", "BM25"])
tabla

,n_queries,MRR,Recall@5,Recall@10
TF-IDF + coseno,34,0.4694,0.5882,0.6176
BM25,34,0.4308,0.5588,0.6471


## 7. Inspección por consulta

In [9]:
filas = []
for rt, rb in zip(r_tfidf, r_bm25):
    filas.append({
        "qid": rt.qid,
        "query": rt.query[:48],
        "rel": ",".join(rt.relevant_ids),
        "rank_TFIDF": rt.first_relevant_rank,
        "rank_BM25": rb.first_relevant_rank,
    })
pd.DataFrame(filas)

,qid,query,rel,rank_TFIDF,rank_BM25
0,q01,Coconut based no bake energy bars with flax seed,recipe_479373,2.0,2.0
1,q02,Fried Jewish fish patties made with ground fish,recipe_60179,1.0,1.0
2,q03,Grilled Middle Eastern style chicken kebabs with,recipe_170046,NaN,NaN
3,q04,Creamy sauce with pasta and sautéed vegetables b,recipe_478609,NaN,NaN
4,q05,Greek-inspired pasta salad with a tangy cheese a,recipe_503003,NaN,NaN
5,q06,Roasted garlic potato recipe with rosemary that',recipe_367050,1.0,1.0
6,q07,Beef and mushroom dish with creamy sauce served,recipe_30159,8.0,7.0
7,q08,Boiled egg salad sandwich recipe with creamy dre,recipe_235097,1.0,2.0
8,q09,Blackened fish recipe with Creole seasoning and,recipe_385182,1.0,1.0
9,q10,baked puff pastry cups filled with creamy aspara,recipe_455645,NaN,NaN


## 8. Análisis de fallos

Consultas en las que **ambos** métodos fallan (el documento relevante no entra en
el top-K). El enunciado pide analizarlas explícitamente.

In [10]:
from src.evaluation import failure_analysis
fails = failure_analysis({"TF-IDF": r_tfidf, "BM25": r_bm25}, k=max(KS))
print(f"Consultas que fallan en ambos (top-{max(KS)}): {len(fails)}\n")
for f in fails:
    print(f["qid"], "|", f["query"])
    print("   relevante:", f["relevant_ids"], "| rank:", f["first_relevant_rank"])
    print("   -> probable causa: brecha léxica (sinónimos/OOV no presentes en el documento)\n")

Consultas que fallan en ambos (top-10): 12

q03 | Grilled Middle Eastern style chicken kebabs with vegetables marinating in a spicy citrus sauce
   relevante: ['recipe_170046'] | rank: {'TF-IDF': None, 'BM25': None}
   -> probable causa: brecha léxica (sinónimos/OOV no presentes en el documento)

q04 | Creamy sauce with pasta and sautéed vegetables baked with extra cheese topping
   relevante: ['recipe_478609'] | rank: {'TF-IDF': None, 'BM25': None}
   -> probable causa: brecha léxica (sinónimos/OOV no presentes en el documento)

q05 | Greek-inspired pasta salad with a tangy cheese and olive twist
   relevante: ['recipe_503003'] | rank: {'TF-IDF': None, 'BM25': None}
   -> probable causa: brecha léxica (sinónimos/OOV no presentes en el documento)

q10 | baked puff pastry cups filled with creamy asparagus and chicken mixture
   relevante: ['recipe_455645'] | rank: {'TF-IDF': None, 'BM25': None}
   -> probable causa: brecha léxica (sinónimos/OOV no presentes en el documento)

q13 | Smoot

## 9. Conclusiones y siguiente paso

- Los baselines **dispersos** (TF-IDF y BM25) recuperan por **solape léxico**: funcionan
  bien cuando la consulta comparte términos con el documento, y fallan ante **sinónimos
  o vocabulario distinto** (ver §8).
- Estas métricas (**Recall@K**, **MRR**) son la **referencia** con la que se compara el
  sistema **RAG** del Hito 2 (recuperación **densa** con *embeddings* y base de datos
  vectorial).
- **Siguiente paso (Hito 2)**: fragmentación + *sentence embeddings* + base vectorial
  (pgvector/FAISS/Chroma), evaluada con **el mismo `queries.jsonl`** para una comparación
  justa, y primera evaluación de la **generación** con un LLM de pesos abiertos.

### Cómo usar vuestro corpus real
1. Prepara un JSONL `{"doc_id","text"}` (o una carpeta de `.txt`) y apunta `CORPUS_PATH`.
2. Genera `queries.jsonl` con la celda §5.b (`RUN_LLM=True`), revisa ≥10 a mano y documenta.
3. Reejecuta el notebook: `Kernel > Restart & Run All`.
